# How robust is the formula if the data is not GARCH?

Generate data from a non-GARCH model, e.g., GJR-GARCH, EGARCH or stochastic volatility, 
with "typical values" of the parameters, and check if the variance of the Sharpe ratio 
is close to that given by the formula in the paper.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import ray
from tqdm.auto import tqdm
from functions import garch_returns, formula_15, estimate_parameters_fast
from functions import legend_thick, remove_scientific_notation_from_vertical_axis
from functions import gjr_garch_returns, egarch_returns, stochastic_volatility_returns

In [ ]:
TIME = 100                       # Number of time points (time series lengths)
REPLICATIONS = 1000              # Number of time series to simulate for each time series length
SAMPLE = True                    # Whether to sample "typical" values of the parameters or use the hard-coded ones
SEEDS = 1 if not SAMPLE else 10  # Number of plots to generate (one for each seed)

In [ ]:
os.makedirs( "plots", exist_ok = True )

In [ ]:
ray.init(
    log_to_driver = False,  # Suppress output (otherwise, the notebook becomes too large)
)

In [ ]:
filename = 'cache/parameters_stocks.parquet'
if os.path.exists(filename) and SAMPLE:
    parameters = pd.read_parquet( filename )
else:
    parameters = pd.DataFrame( [ { 
        'SR': 0.07,
        'mu': 0.002,
        'sigma': 0.03,
        'alpha': 0.06,
        'beta': 0.87,
        'phi': 0.93,
        'skew': -0.54,
        'kurtosis': 5.43,
        'skew_t_a': 1.7,
        'skew_t_b': 1.6,
        'skew_t_loc': -0.1,
        'skew_t_scale': 0.6,
        
        'gjr_omega': .01,
        'gjr_alpha': 0,
        'gjr_gamma': .15,
        'gjr_beta': .9,
        'egarch_omega': 0,
        'egarch_alpha': .25,
        'egarch_beta': .9,
        'egarch_gamma': -.1,
        'sv_mu': 0,
        'sv_phi': .93,
        'sv_sigma_v': 0.25,
    } ] )
    SEEDS = 1

columns = [
    'SR', 
    'mu', 'sigma', 'skew', 'kurtosis', 
    'alpha', 'beta', 'phi', 
    #'T', 
    #'denominator', 
    'gjr_omega', 'gjr_alpha', 'gjr_gamma', 'gjr_beta',
    'egarch_omega', 'egarch_alpha', 'egarch_gamma', 'egarch_beta', 
    'sv_mu', 'sv_phi', 'sv_sigma_v', 
]
parameters[columns].round(3)

In [ ]:
if parameters.shape[0] > 1:

    fig, axs = plt.subplots( 1, 4, figsize = (12,2), layout = 'constrained', dpi = 300 )
    for column, ax in zip( ['gjr_omega', 'gjr_alpha', 'gjr_gamma', 'gjr_beta'], axs ):
        ax.hist( parameters[column], bins = 100, color = 'lightblue',density = True )
        ax.set_xlabel( column )
        for side in ['top', 'right', 'left']:
            ax.spines[side].set_visible(False)
        ax.set_yticks([])
    plt.show()

    fig, axs = plt.subplots( 1, 4, figsize = (12,2), layout = 'constrained', dpi = 300 )
    for column, ax in zip( ['egarch_omega', 'egarch_alpha', 'egarch_gamma', 'egarch_beta'], axs ):
        ax.hist( 
            np.clip( 
                parameters[column], 
                -2, 2,
            ), 
            bins = 100, color = 'lightblue', density = True,
        )
        ax.set_xlabel( column )
        for side in ['top', 'right', 'left']:
            ax.spines[side].set_visible(False)
        ax.set_yticks([])
    plt.show()

In [ ]:
@ray.remote
def f(row, Ts, model):
    assert model in ['garch','gjr', 'egarch', 'sv']
    SRs = []
    if model == 'garch':
        params = row['mu'], row['sigma'], row['alpha'], row['beta']
        f = garch_returns
    elif model == 'gjr':
        params = row['mu'], row['sigma'], row['gjr_omega'], row['gjr_alpha'], row['gjr_gamma'], row['gjr_beta']
        f = gjr_garch_returns
    elif model == 'egarch':
        params = row['mu'], row['sigma'], row['egarch_omega'], row['egarch_alpha'], row['egarch_gamma'], row['egarch_beta']
        params = [ np.clip(p, -2, 2) for p in params ]
        f = egarch_returns
    elif model == 'sv':
        params = row['mu'], row['sigma'], row['sv_mu'], row['sv_phi'], row['sv_sigma_v']
        f = stochastic_volatility_returns
    for T in Ts:
        innovations = scipy.stats.jf_skew_t.rvs(row['skew_t_a'], row['skew_t_b'], row['skew_t_loc'], row['skew_t_scale'], size = T + 100)
        
        y = f(T, *params, innovations)[0]
        #y = ( y - y.mean() ) / y.std()
        #y = row['mu'] + y * row['sigma']

        par = estimate_parameters_fast( y ) 
        
        var_theoretical = formula_15( 
            SR = par.SR,
            skew = par.skew, 
            kurtosis = par.kurtosis, 
            alpha = par.alpha, 
            beta = par.beta, 
            T = T
        )

        SRs.append( { 
            'T': T,
            'var_theoretical': var_theoretical,
            'SR': y.mean() / y.std(),
        } )

    SRs = pd.DataFrame( SRs )
    return SRs

for SEED in range(SEEDS): 
    for MODEL in ['garch', 'gjr', 'egarch', 'sv']:

        np.random.seed(SEED)
        row = parameters.sample(1).iloc[0,:]
        
        if MODEL == 'garch':
            if not( 
                row['alpha'] >= 0
                and row['beta'] >= 0
                and row['alpha'] + row['beta'] < 1
            ):
                continue
            print( f"GARCH model: alpha={row['alpha']:.3f}, beta={row['beta']:.3f}" )
        elif MODEL == 'gjr': 
            if not( 
                row['gjr_alpha'] >= 0
                and row['gjr_beta'] >= 0
                and row['gjr_alpha'] + row['gjr_beta'] < 1
            ):
                continue
            print( f"GJR-GARCH model: omega={row['gjr_omega']:.3f}, alpha={row['gjr_alpha']:.3f}, gamma={row['gjr_gamma']:.3f}, beta={row['gjr_beta']:.3f}" )

        if MODEL == 'egarch':
            if not( 
                row['egarch_alpha'] >= 0
                and row['egarch_beta'] >= 0
                #and row['egarch_alpha'] + row['egarch_beta'] < 1
            ):
                continue
            print( f"EGARCH model: omega={row['egarch_omega']:.3f}, alpha={row['egarch_alpha']:.3f}, gamma={row['egarch_gamma']:.3f}, beta={row['egarch_beta']:.3f}" )

        if MODEL == 'sv':
            # TODO: Any condition to check?
            print( f"Stochastic volatility model: mu={row['sv_mu']:.3f}, phi={row['sv_phi']:.3f}, sigma_v={row['sv_sigma_v']:.3f}" )


        # To make sure the time series look file, generate one and plot it
        if MODEL == 'garch':
            params = row['mu'], row['sigma'], row['alpha'], row['beta']
            F = garch_returns
        elif MODEL == 'gjr':
            params = row['mu'], row['sigma'], row['gjr_omega'], row['gjr_alpha'], row['gjr_gamma'], row['gjr_beta']
            F = gjr_garch_returns
        elif MODEL == 'egarch':
            params = row['mu'], row['sigma'], row['egarch_omega'], row['egarch_alpha'], row['egarch_gamma'], row['egarch_beta']
            params = [ np.clip(p, -2, 2) for p in params ]
            F = egarch_returns
        elif MODEL == 'sv':
            params = row['mu'], row['sigma'], row['sv_mu'], row['sv_phi'], row['sv_sigma_v']
            F = stochastic_volatility_returns
        innovations = scipy.stats.jf_skew_t.rvs(row['skew_t_a'], row['skew_t_b'], row['skew_t_loc'], row['skew_t_scale'], size = 2100)
        y, sd = F(2000, *params, innovations)
        fig, ax = plt.subplots( figsize = (15,3), layout = 'constrained')
        ax.plot(y)
        ax.plot(sd)
        ax.set_xlim( 0, len(y) )
        plt.show()
        
        Ts = np.unique( np.logspace( 1, 3, TIME ).astype(int) )  # Was: 100
        SRs = [ f.remote(row, Ts, MODEL) for _ in range(REPLICATIONS) ]
        SRs = [ ray.get(u) for u in tqdm(SRs) ]
        SRs = pd.concat( SRs )

        fig, ax = plt.subplots( figsize = ( 6,3 ), layout = 'constrained', dpi = 300 )
        ax.scatter( 
            SRs['T'], 
            SRs['var_theoretical'], 
            color = 'tab:orange', 
            label = "Theoretical",
            s = 20,
            alpha = 1/255,
        )
        ax.plot( SRs.groupby('T')['SR'].var(), color = 'tab:blue', label = "Sample" )
        ax.set_xlabel('Number of observations')
        ax.set_ylabel('Variance of SR')
        ax.set_xscale('log')
        leg = ax.legend()
        for lh in leg.legend_handles:
            lh.set_alpha(1)
        #df_text = f"{row['nct_df']:.1f}" if row['nct_df'] < 100 else r"$\infty$"
        if MODEL == 'garch':
            title = ( 
                f"SR={row['SR']:.2f}"
                f"   skew={row['skew']:.2f}   kurtosis={row['kurtosis']:.2f}"
                "\nGARCH:"
                r"   $\alpha=$" f"{row['alpha']:.2f}"
                r"   $\beta=$" f"{row['beta']:.2f}"
            )
        elif MODEL == 'gjr':
            title = ( 
                f"SR={row['SR']:.2f}"
                f"   skew={row['skew']:.2f}   kurtosis={row['kurtosis']:.2f}"
                "\nGJR-GARCH:"
                r"   $\alpha=$" f"{row['gjr_alpha']:.2f}" 
                r"   $\beta=$" f"{row['gjr_beta']:.2f}" 
                r"   $\gamma=$" f"{row['gjr_gamma']:.2f}"
            )
        elif MODEL == 'egarch':
            title = ( 
                f"SR={row['SR']:.2f}"
                f"   skew={row['skew']:.2f}   kurtosis={row['kurtosis']:.2f}"
                "\nEGARCH:"
                #r"   $\omega=$" f"{row['egarch_omega']:.2f}"
                r"   $\alpha=$" f"{row['egarch_alpha']:.2f}"
                r"   $\gamma=$" f"{row['egarch_gamma']:.2f}"
                r"   $\beta=$" f"{row['egarch_beta']:.2f}"
            )
        elif MODEL == 'sv':
            title = ( 
                f"SR={row['SR']:.2f}"
                f"   skew={row['skew']:.2f}   kurtosis={row['kurtosis']:.2f}"
                "\nStochastic volatility:"
                r"   $\mu=$" f"{row['sv_mu']:.2f}"
                r"   $\phi=$" f"{row['sv_phi']:.2f}"
                r"   $\sigma_v=$" f"{row['sv_sigma_v']:.2f}"
            )
        ax.set_title( title )
        ax.set_ylim( -.01, .2 )
        ax.axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
        if SAMPLE: 
            filename = f"plots/plot_1__{MODEL}_R={REPLICATIONS}__T={len(Ts)}__SEED={SEED}"
        else: 
            filename = f"plots/plot_1__{MODEL}_R={REPLICATIONS}__T={len(Ts)}"
        plt.savefig( f"{filename}.png", dpi = 300 )
        plt.show()


        tmp = pd.merge( 
            SRs.groupby('T')['SR'].var().reset_index(),
            SRs.set_index('T')[['var_theoretical']].reset_index(),
            on = 'T',
        ).set_index('T').rename( columns = { 'SR': 'var(SR)' } )
        tmp['var(SR)/var(theoretical)'] = tmp['var(SR)'] / tmp['var_theoretical']
        fig, ax = plt.subplots( figsize = ( 6,3 ), layout = 'constrained', dpi = 300 )
        ax.scatter( tmp.index, tmp['var(SR)/var(theoretical)'], alpha = .01 )
        ax.axhline( 1, color = 'black', linestyle = ':', linewidth = 1 )
        ax.set_ylim( 0, 2 )
        ax.set_ylim(.1, 10); ax.set_yscale( 'log' )
        ax.set_xscale( 'log' )
        ax.set_xlabel( 'Number of observations' )
        ax.set_ylabel( 'Variance of SR / Theoretical variance' )
        ax.set_title( title )
        plt.show()
        